In [ ]:
"""Day 16 / Session 4 - Lab Assignment 1: K-Means from Scratch - Lloyd's Algorithm in numpy."""
import numpy as np
from sklearn.datasets import make_blobs


def make_data(seed=42):
    return make_blobs(n_samples=300, centers=4, n_features=2, cluster_std=1.0, random_state=seed)


def euclidean(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    return float(np.sqrt(np.sum((p - q) ** 2)))


def assign_clusters(X, centroids):
    X, centroids = np.asarray(X, dtype=float), np.asarray(centroids, dtype=float)
    labels = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        labels[i] = int(np.argmin([euclidean(x, c) for c in centroids]))
    return labels


def update_centroids(X, labels, k):
    X, labels = np.asarray(X, dtype=float), np.asarray(labels)
    out = np.full((k, X.shape[1]), np.nan)
    for j in range(k):
        members = X[labels == j]
        if len(members):
            out[j] = members.mean(axis=0)
    return out  # an empty cluster stays NaN here; kmeans() keeps the old centroid for it


def compute_inertia(X, labels, centroids):
    X, labels, centroids = np.asarray(X, dtype=float), np.asarray(labels), np.asarray(centroids, dtype=float)
    return float(np.sum((X - centroids[labels]) ** 2))


def kmeans(X, k, init_centroids, max_iter=100, tol=1e-6):
    X = np.asarray(X, dtype=float)
    centroids = np.asarray(init_centroids, dtype=float).copy()
    labels = assign_clusters(X, centroids)
    for _ in range(max_iter):
        new = update_centroids(X, labels, k)
        new = np.where(np.isnan(new), centroids, new)          # keep the old centroid if a cluster went empty
        shift = np.max(np.linalg.norm(new - centroids, axis=1))
        centroids = new
        labels = assign_clusters(X, centroids)
        if shift < tol:
            break
    return labels, centroids, compute_inertia(X, labels, centroids)